In [1]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
import os

In [2]:

index_df = pd.read_csv(
    r"D:\DACN\dbl_alphapepdeep\_lt\251128\_index.tsv",
    sep="\t",
    header=None
)

index_df.head()


,0
0,1/17705.tsv
1,1/27289.tsv
2,1/26084.tsv
3,1/7517.tsv
4,1/26511.tsv


In [3]:
base_dir = r"D:\DACN\dbl_alphapepdeep\_lt\251128\251128"

dfs = []

for rel_path in index_df[0]:
    full_path = os.path.join(base_dir, rel_path)
    
    df = pd.read_csv(full_path, sep="\t")
    dfs.append(df)


In [4]:
df = pd.concat(dfs, ignore_index=True)

print(df.head())
print(f"Total rows: {len(df)}")

   PrecursorCharge            ModifiedPeptide  ... FragmentCharge  RelativeIntensity
0                2  _AEEDEILNRS(UniMod:21)PR_  ...              1           0.085714
1                2  _AEEDEILNRS(UniMod:21)PR_  ...              1           1.000000
2                2  _AEEDEILNRS(UniMod:21)PR_  ...              1           0.028571
3                2  _AEEDEILNRS(UniMod:21)PR_  ...              1           0.014286
4                2  _AEEDEILNRS(UniMod:21)PR_  ...              1           0.057143

[5 rows x 7 columns]
Total rows: 35702


In [5]:
colums = df.columns.tolist()
colums 

['PrecursorCharge',
 'ModifiedPeptide',
 'FragmentLossType',
 'FragmentNumber',
 'FragmentType',
 'FragmentCharge',
 'RelativeIntensity']

In [6]:
import re
df_peptide = df[['ModifiedPeptide', 'PrecursorCharge']].drop_duplicates()
df_peptide[0:10]

,ModifiedPeptide,PrecursorCharge
0,_AEEDEILNRS(UniMod:21)PR_,2
625,_AGDLLEDS(UniMod:21)PKRPK_,2
1201,_AGLGS(UniMod:21)PERPPK_,2
1585,_AHT(UniMod:21)PTPGIYMGRPTHSGGGGGGGGGGGGGGGGR_,4
2556,_AKS(UniMod:21)PQPPVEEEDEHFDDTVVC(UniMod:4)LDT...,3
3835,_ALASEKS(UniMod:21)PTADAKPAPK_,2
4340,_ALSAS(UniMod:21)HTDLAH_,2
4774,_ANGQENGHVKS(UniMod:21)NGDLSPK_,2
5421,_ANS(UniMod:21)PEKPPEAGAAHKPR_,3
6074,_ASAVSELS(UniMod:21)PR_,2


In [7]:
def parse_modpep(modpep):
    import re
    pep = modpep.strip('_')
    seq = ""
    mods = []
    mod_sites = []
    

    unimod_mapping = {
        "1": "Acetyl",
        "4": "Carbamidomethyl",
        "21": "Phospho",
        "35": "Oxidation"
    }
    
    i = 0
    while i < len(pep):
     
        if i == 0 and pep.startswith("(UniMod:"):
            m_nterm = re.match(r"\(UniMod:(\d+)\)", pep)
            if m_nterm:
                uid = m_nterm.group(1)
                base_name = unimod_mapping.get(uid, f"Acetyl") 
                # N-term mod trong PeptDeep có định dạng Tên@Any_N-term và site là 0
                mods.append(f"{base_name}@Any_N-term")
                mod_sites.append("0")
                i += len(m_nterm.group(0))
                continue

       
        m = re.match(r"([A-Z])\(UniMod:(\d+)\)", pep[i:])
        if m:
            aa = m.group(1)
            uid = m.group(2)
            base_name = unimod_mapping.get(uid, f"UniMod{uid}") 
            mods.append(f"{base_name}@{aa}")
            
            # Vị trí acid amin bắt đầu từ 1
            mod_sites.append(str(len(seq) + 1)) 
            
            seq += aa
            i += len(m.group(0))
        else:
            if pep[i].isalpha():
                seq += pep[i]
            i += 1
            
    return seq, ";".join(mods), ";".join(mod_sites)

In [8]:
df_peptide[['sequence', 'mods', 'mod_sites']] = df_peptide['ModifiedPeptide'].apply(
    lambda x: pd.Series(parse_modpep(x))
)
df_peptide["nAA"] = df_peptide['sequence'].apply(len)
df_peptide[0:10]

,ModifiedPeptide,PrecursorCharge,sequence,mods,mod_sites,nAA
0,_AEEDEILNRS(UniMod:21)PR_,2,AEEDEILNRSPR,Phospho@S,10,12
625,_AGDLLEDS(UniMod:21)PKRPK_,2,AGDLLEDSPKRPK,Phospho@S,8,13
1201,_AGLGS(UniMod:21)PERPPK_,2,AGLGSPERPPK,Phospho@S,5,11
1585,_AHT(UniMod:21)PTPGIYMGRPTHSGGGGGGGGGGGGGGGGR_,4,AHTPTPGIYMGRPTHSGGGGGGGGGGGGGGGGR,Phospho@T,3,33
2556,_AKS(UniMod:21)PQPPVEEEDEHFDDTVVC(UniMod:4)LDT...,3,AKSPQPPVEEEDEHFDDTVVCLDTYNCDLHFK,Phospho@S;Carbamidomethyl@C;Carbamidomethyl@C,3;21;27,32
3835,_ALASEKS(UniMod:21)PTADAKPAPK_,2,ALASEKSPTADAKPAPK,Phospho@S,7,17
4340,_ALSAS(UniMod:21)HTDLAH_,2,ALSASHTDLAH,Phospho@S,5,11
4774,_ANGQENGHVKS(UniMod:21)NGDLSPK_,2,ANGQENGHVKSNGDLSPK,Phospho@S,11,18
5421,_ANS(UniMod:21)PEKPPEAGAAHKPR_,3,ANSPEKPPEAGAAHKPR,Phospho@S,3,17
6074,_ASAVSELS(UniMod:21)PR_,2,ASAVSELSPR,Phospho@S,8,10


In [9]:
df_peptide.head()

,ModifiedPeptide,PrecursorCharge,sequence,mods,mod_sites,nAA
0,_AEEDEILNRS(UniMod:21)PR_,2,AEEDEILNRSPR,Phospho@S,10,12
625,_AGDLLEDS(UniMod:21)PKRPK_,2,AGDLLEDSPKRPK,Phospho@S,8,13
1201,_AGLGS(UniMod:21)PERPPK_,2,AGLGSPERPPK,Phospho@S,5,11
1585,_AHT(UniMod:21)PTPGIYMGRPTHSGGGGGGGGGGGGGGGGR_,4,AHTPTPGIYMGRPTHSGGGGGGGGGGGGGGGGR,Phospho@T,3,33
2556,_AKS(UniMod:21)PQPPVEEEDEHFDDTVVC(UniMod:4)LDT...,3,AKSPQPPVEEEDEHFDDTVVCLDTYNCDLHFK,Phospho@S;Carbamidomethyl@C;Carbamidomethyl@C,3;21;27,32


In [10]:
df_peptide = df_peptide.rename(columns={'PrecursorCharge': 'charge'})
# df_peptide.drop(columns=[ 'mod_sites'], inplace=True) 

In [56]:
df_peptide[0:10]

,ModifiedPeptide,charge,sequence,mods,nAA
0,_AEEDEILNRS(UniMod:21)PR_,2,AEEDEILNRSPR,Phospho@S,12
625,_AGDLLEDS(UniMod:21)PKRPK_,2,AGDLLEDSPKRPK,Phospho@S,13
1201,_AGLGS(UniMod:21)PERPPK_,2,AGLGSPERPPK,Phospho@S,11
1585,_AHT(UniMod:21)PTPGIYMGRPTHSGGGGGGGGGGGGGGGGR_,4,AHTPTPGIYMGRPTHSGGGGGGGGGGGGGGGGR,Phospho@T,33
2556,_AKS(UniMod:21)PQPPVEEEDEHFDDTVVC(UniMod:4)LDT...,3,AKSPQPPVEEEDEHFDDTVVCLDTYNCDLHFK,Phospho@S;Carbamidomethyl@C;Carbamidomethyl@C,32
3835,_ALASEKS(UniMod:21)PTADAKPAPK_,2,ALASEKSPTADAKPAPK,Phospho@S,17
4340,_ALSAS(UniMod:21)HTDLAH_,2,ALSASHTDLAH,Phospho@S,11
4774,_ANGQENGHVKS(UniMod:21)NGDLSPK_,2,ANGQENGHVKSNGDLSPK,Phospho@S,18
5421,_ANS(UniMod:21)PEKPPEAGAAHKPR_,3,ANSPEKPPEAGAAHKPR,Phospho@S,17
6074,_ASAVSELS(UniMod:21)PR_,2,ASAVSELSPR,Phospho@S,10


In [ ]:
from peptdeep.pretrained_models import ModelManager
model_mgr = ModelManager(device='cpu')
ms2_pred = model_mgr.predict_ms2(df_peptide[0:1]) 
# rt_df = model_mgr.predict_rt(df_peptide)
# print(rt_df.head())
# print(rt_df.columns)


ImportError: cannot import name 'ModelManager' from 'peptdeep.model.ms2' (D:\DACN\dbl_alphapepdeep\alphapeptdeep\peptdeep\model\ms2.py)

: 

In [ ]:
total_params = sum(p.numel() for p in model_mgr.parameters())
trainable_params = sum(p.numel() for p in model_mgr.parameters() if p.requires_grad)

AttributeError: 'ModelManager' object has no attribute 'parameters'

In [ ]:
print(dir(model_mgr))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_instrument', '_predict_func_for_mp', '_train_psm_logging', 'batch_size_to_train_charge', 'batch_size_to_train_ms2', 'batch_size_to_train_rt_ccs', 'ccs_model', 'charge_model', 'charge_prob_cutoff', 'epoch_to_train_charge', 'epoch_to_train_ms2', 'epoch_to_train_rt_ccs', 'instrument', 'load_external_models', 'load_installed_models', 'lr_to_train_charge', 'lr_to_train_ms2', 'lr_to_train_rt_ccs', 'ms2_model', 'nce', 'predict_all', 'predict_all_mp', 'predict_charge', 'predict_mobility', 'predict_ms2', 'predict_rt', 'psm_num_per_mod_to_train_charge', 'psm_num_per_mod_to_train_ms2', 'psm_num_per_mod_to_train_rt_ccs', 'psm_num_to_test_charge', 'psm_num_t

In [ ]:
ccs_pred = model_mgr.predict_mobility(df_peptide)
print(ccs_pred.head())

In [ ]:
ms2_pred = model_mgr.predict_ms2(df_peptide[0:1])
print(ms2_pred)

2026-01-30 20:38:21> Predicting MS2 ...


D:\DACN\dbl_alphapepdeep\alphapeptdeep\peptdeep\model\ms2.py:465: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  precursor_df.drop(
c:\Users\letie\anaconda3\envs\peptdeep_env\lib\site-packages\alphabase\peptide\fragment.py:534: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  precursor_df["frag_start_idx"] = start_indices
c:\Users\letie\anaconda3\envs\peptdeep_env\lib\site-packages\alphabase\peptide\fragment.py:535: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See 

        b_z1  b_z2      y_z1      y_z2  b_modloss_z1  b_modloss_z2  \
0   0.000000   0.0  0.000000  0.000000      0.000000           0.0   
1   0.119571   0.0  0.000000  0.057406      0.000000           0.0   
2   0.021620   0.0  0.000000  0.045024      0.000000           0.0   
3   0.038291   0.0  0.000000  0.098820      0.000000           0.0   
4   0.071498   0.0  0.000000  0.035853      0.000000           0.0   
5   0.050915   0.0  0.053717  0.000000      0.000000           0.0   
6   0.009950   0.0  0.042173  0.000000      0.000000           0.0   
7   0.000000   0.0  0.002873  0.000000      0.000000           0.0   
8   0.065352   0.0  0.020741  0.000000      0.000000           0.0   
9   0.105977   0.0  1.000000  0.000000      0.886432           0.0   
10  0.000000   0.0  0.116241  0.000000      0.028951           0.0   

    y_modloss_z1  y_modloss_z2  
0       0.000000      0.024195  
1       0.003119      0.192764  
2       0.010513      0.099997  
3       0.000000      0.289

In [ ]:
df_peptide.shape

(54, 10)

In [ ]:
ms2_pred.shape

(954, 8)

In [ ]:
model_mgr.pearson_correlation_ms2(df_peptide, ms2_pred)

In [ ]:
df_peptide[0:10]

,ModifiedPeptide,charge,sequence,mods,mod_sites,nAA,nce,instrument,frag_start_idx,frag_stop_idx
0,_AEEDEILNRS(UniMod:21)PR_,2,AEEDEILNRSPR,Phospho@S,10,12,30.0,Lumos,0,11
625,_AGDLLEDS(UniMod:21)PKRPK_,2,AGDLLEDSPKRPK,Phospho@S,8,13,30.0,Lumos,11,23
1201,_AGLGS(UniMod:21)PERPPK_,2,AGLGSPERPPK,Phospho@S,5,11,30.0,Lumos,23,33
1585,_AHT(UniMod:21)PTPGIYMGRPTHSGGGGGGGGGGGGGGGGR_,4,AHTPTPGIYMGRPTHSGGGGGGGGGGGGGGGGR,Phospho@T,3,33,30.0,Lumos,33,65
2556,_AKS(UniMod:21)PQPPVEEEDEHFDDTVVC(UniMod:4)LDT...,3,AKSPQPPVEEEDEHFDDTVVCLDTYNCDLHFK,Phospho@S;Carbamidomethyl@C;Carbamidomethyl@C,3;21;27,32,30.0,Lumos,65,96
3835,_ALASEKS(UniMod:21)PTADAKPAPK_,2,ALASEKSPTADAKPAPK,Phospho@S,7,17,30.0,Lumos,96,112
4340,_ALSAS(UniMod:21)HTDLAH_,2,ALSASHTDLAH,Phospho@S,5,11,30.0,Lumos,112,122
4774,_ANGQENGHVKS(UniMod:21)NGDLSPK_,2,ANGQENGHVKSNGDLSPK,Phospho@S,11,18,30.0,Lumos,122,139
5421,_ANS(UniMod:21)PEKPPEAGAAHKPR_,3,ANSPEKPPEAGAAHKPR,Phospho@S,3,17,30.0,Lumos,139,155
6074,_ASAVSELS(UniMod:21)PR_,2,ASAVSELSPR,Phospho@S,8,10,30.0,Lumos,155,164


In [ ]:

peptide_0_ms2 = ms2_pred.iloc[0:11, :] 
print(peptide_0_ms2)

        b_z1  b_z2      y_z1      y_z2  b_modloss_z1  b_modloss_z2  \
0   0.000000   0.0  0.000000  0.000000      0.000000           0.0   
1   0.119571   0.0  0.000000  0.057406      0.000000           0.0   
2   0.021620   0.0  0.000000  0.045024      0.000000           0.0   
3   0.038291   0.0  0.000000  0.098820      0.000000           0.0   
4   0.071498   0.0  0.000000  0.035853      0.000000           0.0   
5   0.050915   0.0  0.053717  0.000000      0.000000           0.0   
6   0.009950   0.0  0.042173  0.000000      0.000000           0.0   
7   0.000000   0.0  0.002873  0.000000      0.000000           0.0   
8   0.065352   0.0  0.020741  0.000000      0.000000           0.0   
9   0.105977   0.0  1.000000  0.000000      0.886432           0.0   
10  0.000000   0.0  0.116241  0.000000      0.028951           0.0   

    y_modloss_z1  y_modloss_z2  
0       0.000000      0.024195  
1       0.003119      0.192764  
2       0.010513      0.099997  
3       0.000000      0.289

In [ ]:
df_peptide['RT_pred'] = rt_df['rt_pred']
df_peptide['CCS_pred'] = ccs_pred['ccs_pred']
df_peptide = pd.concat([df_peptide, ms2_pred], axis=1)
df_peptide

,ModifiedPeptide,charge,sequence,mods,mod_sites,nAA,rt_pred,rt_norm_pred,ccs_pred,precursor_mz,...,RT_pred,CCS_pred,b_z1,b_z2,y_z1,y_z2,b_modloss_z1,b_modloss_z2,y_modloss_z1,y_modloss_z2
0,_AEEDEILNRS(UniMod:21)PR_,2.0,AEEDEILNRSPR,Phospho@S,9,12.0,0.287161,0.287161,414.568115,754.840677,...,0.287161,414.568115,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.010220
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.181293,0.0,0.000000,0.039563,0.000000,0.0,0.000000,0.171757
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.033880,0.0,0.000000,0.038621,0.000000,0.0,0.000000,0.123552
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.068147,0.0,0.000000,0.092495,0.000000,0.0,0.000000,0.231730
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.155229,0.0,0.007308,0.035249,0.000000,0.0,0.012172,0.064771
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.137313,0.0,0.113238,0.000000,0.000000,0.0,0.080364,0.010980
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.046837,0.0,0.102326,0.000000,0.000000,0.0,0.061840,0.000000
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.008668,0.0,0.055006,0.000000,0.000000,0.0,0.056454,0.000639
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.074537,0.0,0.034933,0.000000,0.015730,0.0,0.000000,0.000000
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.024236,0.0,1.000000,0.000000,0.497679,0.0,0.000000,0.000000
